In [ ]:
# 【教学说明】第1周第2天：用 Ollama（OpenAI 兼容端点）做网站尖刻摘要
# 练习目标：抓网页 → messages(system/user) → chat.completions → Markdown 展示
# 环境：.env 里准备 OLLAMA_API_KEY、OLLAMA_BASE_URL；模型名 llama3.2 需本机已拉取
# 和本课关系：同一套 OpenAI SDK 写法，只是 base_url 指向 Ollama 而不是云端 OpenAI
# 怎么跑：确认 scraper 可导入后，整格 Shift+Enter；最后会摘要 https://edwarddonner.com

# ========== 导入 + 环境 + Ollama 客户端 + 摘要流水线（整格可从上到下读） ==========

# 导入标准库 os：读环境变量（Environment Variables）
import os
# 从 dotenv 导入 load_dotenv：把 .env 里的密钥/地址读进环境变量
from dotenv import load_dotenv
# 从 scraper 导入抓取函数：按 URL 取网站正文
from scraper import fetch_website_contents
# 从 IPython.display 导入展示工具：用 Markdown 漂亮显示摘要
from IPython.display import Markdown, display
# 从 openai 导入 OpenAI 客户端：这里用来连 Ollama 的 OpenAI 兼容接口
from openai import OpenAI


# 加载 .env；override=True 表示用文件覆盖进程里已有的同名变量
load_dotenv(override=True)

# 读取 Ollama 侧的 API Key（有的部署需要；名字以本练习 .env 为准）
api_key = os.getenv('OLLAMA_API_KEY')

# 读取 Ollama OpenAI 兼容服务的 base_url（例如 http://localhost:11434/v1）
ollama_base_url = os.getenv("OLLAMA_BASE_URL")

# 检查钥匙：缺失 / 首尾有空白 / 看起来正常
if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")


# 检查基本 URL：缺失 / 首尾有空白 / 看起来正常
if not ollama_base_url:
    print("No base url was found - please confirm that the lamma base url has been set in the dotenv file")
elif ollama_base_url.strip() != ollama_base_url:
    print("A base url was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("Base url found and looks good so far!")


# 创建客户端：请求发往 Ollama 兼容端点，而不是默认的 OpenAI 云
ollamaClient = OpenAI(base_url=ollama_base_url, api_key=api_key)

# 先抓一份示例站正文（变量 ed 保留原样；后面 display_summary 会再抓一次同一 URL）
ed = fetch_website_contents("https://edwarddonner.com")

# system prompt：尖酸幽默摘要；忽略导航文字；纯 Markdown（发给模型的指令不翻译）
system_prompt = """
You are a snarky assistant that analyzes the contents of a website,
and provides a short, snarky, humorous summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

# user prompt 前缀：任务说明；真正正文在 messages_for 里拼接
user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.

"""

# 组装 Chat Completions 所需的 messages 列表
def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website}
    ]


# 抓取 URL → 调本地 llama3.2 → 返回摘要字符串
def summarize(url):
    # 抓取网站正文
    website = fetch_website_contents(url)
    # 用 Ollama 兼容客户端发起非流式聊天补全
    response = ollamaClient.chat.completions.create(
        model = "llama3.2",
        messages = messages_for(website)
    )
    # 取出第一条 choice 的文本
    return response.choices[0].message.content



# 摘要并把结果渲染成 Markdown
def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))


# 试跑：摘要课程作者主页（URL 保持原样）
display_summary("https://edwarddonner.com")


